# Pre-Processing

## 1. Import libraries, get data and create the dataframe

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [3]:
data_path = Path.cwd() / ".." / "data" / "processed" / "healthcare_dataset_semi-processed.csv"
data = pd.read_csv(data_path)

In [4]:
data.head()

,Age,Gender,Blood Type,Medical Condition,Insurance Provider,Admission Type,Medication,Test Results,LOS Category
0,19,Female,AB+,Infections,Blue Cross,Emergency,Azithromycin,Normal,Medium Stay
1,15,Female,B-,Flu,UnitedHealthcare,Emergency,Tamiflu,Abnormal,Short Stay
2,50,Female,A+,Cancer,Blue Cross,Elective,Cisplatin,Inconclusive,Long Stay
3,24,Female,O+,Asthma,Aetna,Elective,Prednisone,Normal,Short Stay
4,80,Female,A+,Heart Disease,Cigna,Routine,Beta-blockers,Inconclusive,Long Stay


In [5]:
data.nunique()

Age                   86
Gender                 2
Blood Type             8
Medical Condition      8
Insurance Provider     5
Admission Type         4
Medication            23
Test Results           3
LOS Category           3
dtype: int64

## 2. Pre-process data

### 2.1 Transform gender in intenger

In [6]:
print(data['Gender'].value_counts())

Gender
Female    27955
Male      27545
Name: count, dtype: int64


In [7]:
data['Gender'] = data['Gender'].map({'Male': 0, 'Female': 1})


### 2.2 One-Hot Encoding + dropping first column to evict dummy variable trap

In [8]:
data = pd.get_dummies(data, columns=['Blood Type', 'Medical Condition', 'Admission Type', 'Medication', 'Test Results'], drop_first=True, dtype=int)

In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 45 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   Age                              55500 non-null  int64
 1   Gender                           55500 non-null  int64
 2   Insurance Provider               55500 non-null  str  
 3   LOS Category                     55500 non-null  str  
 4   Blood Type_A-                    55500 non-null  int64
 5   Blood Type_AB+                   55500 non-null  int64
 6   Blood Type_AB-                   55500 non-null  int64
 7   Blood Type_B+                    55500 non-null  int64
 8   Blood Type_B-                    55500 non-null  int64
 9   Blood Type_O+                    55500 non-null  int64
 10  Blood Type_O-                    55500 non-null  int64
 11  Medical Condition_Asthma         55500 non-null  int64
 12  Medical Condition_Cancer         55500 non-null  int64
 1

In [10]:
data.head()

,Age,Gender,Insurance Provider,LOS Category,Blood Type_A-,Blood Type_AB+,Blood Type_AB-,Blood Type_B+,Blood Type_B-,Blood Type_O+,...,Medication_Orlistat,Medication_Oseltamivir,Medication_Phentermine,Medication_Prednisone,Medication_Rivastigmine,Medication_Statins,Medication_Tamiflu,Medication_Zanamivir,Test Results_Inconclusive,Test Results_Normal
0,19,1,Blue Cross,Medium Stay,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,15,1,UnitedHealthcare,Short Stay,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
2,50,1,Blue Cross,Long Stay,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,24,1,Aetna,Short Stay,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,1
4,80,1,Cigna,Long Stay,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


### 2.3 Create new dataframe without insurance provider 
I'll fit both data, with and without insurance provider to compare results

In [11]:
data_without_insurance = data.drop(columns=['Insurance Provider'])

In [12]:
data = pd.get_dummies(data, columns=['Insurance Provider'], drop_first=True, dtype=int)

In [13]:
data.head()

,Age,Gender,LOS Category,Blood Type_A-,Blood Type_AB+,Blood Type_AB-,Blood Type_B+,Blood Type_B-,Blood Type_O+,Blood Type_O-,...,Medication_Rivastigmine,Medication_Statins,Medication_Tamiflu,Medication_Zanamivir,Test Results_Inconclusive,Test Results_Normal,Insurance Provider_Blue Cross,Insurance Provider_Cigna,Insurance Provider_Medicare,Insurance Provider_UnitedHealthcare
0,19,1,Medium Stay,0,1,0,0,0,0,0,...,0,0,0,0,0,1,1,0,0,0
1,15,1,Short Stay,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,1
2,50,1,Long Stay,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
3,24,1,Short Stay,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,80,1,Long Stay,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0


## 3. Export processed data

In [ ]:
data.to_csv(Path.cwd() / ".." / "data" / "processed" / "healthcare_dataset_processed.csv", index=False)
data_without_insurance.to_csv(Path.cwd() / ".." / "data" / "processed" / "processed_dataset_without_insurance.csv", index=False)
pd.to_csv(data_without_insurance, Path.cwd() / ".." / "data" / "processed" / "processed_dataset_without_insurance.csv"), index=False)

## 4. Split data by train and test

Variables


In [18]:
y = data['LOS Category']
X1 = data.drop(columns=['LOS Category'])
X2 = data_without_insurance.drop(columns=['LOS Category'])

Splitting

In [19]:
train_X1, test_X1, train_y, test_y = train_test_split(X1, y, test_size=0.2, random_state=42, stratify=y)

Fitting

In [21]:
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(train_X1, train_y)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current